# Module 7 · Hallucination, Context Limits & Best-of-N
Understand and mitigate hallucination. Handle context window limits. Generate diverse responses.

---
> **Setup:** Run the first two cells before anything else.
> API key is loaded automatically from the `.env` file in the parent folder.

In [1]:
%pip install -q google-genai python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, json, math, time, base64, getpass, re, urllib.request
from dotenv import load_dotenv
from google import genai
from google.genai import types

load_dotenv()  # picks up modules/.env symlink
api_key = os.environ.get('GEMINI_API_KEY')
if not api_key:
    api_key = getpass.getpass('Paste your Gemini API key: ')

client = genai.Client(api_key=api_key)
MODEL = 'gemma-4-31b-it'
DEFAULT_MAX = 2048

def cfg(**kwargs):
    kwargs.setdefault('max_output_tokens', DEFAULT_MAX)
    kwargs.setdefault('temperature', 0.7)
    return types.GenerateContentConfig(**kwargs)

def get_text(response) -> str:
    if response.text:
        return response.text.strip()
    parts = []
    for cand in (response.candidates or []):
        if cand.content:
            for part in cand.content.parts:
                if not getattr(part, 'thought', False) and part.text:
                    parts.append(part.text)
    return ''.join(parts).strip()

def _call_with_retry(fn, *args, max_retries=5, **kwargs):
    for attempt in range(max_retries):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            msg = str(e)
            if '429' in msg or 'RESOURCE_EXHAUSTED' in msg:
                # parse suggested retry delay from error (e.g. 'retryDelay': '30s')
                m = re.search(r'retry[^0-9]*([0-9]+)s', msg, re.I)
                wait = int(m.group(1)) + 5 if m else 35
                print(f'  ⏳ Rate-limited — waiting {wait}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                raise
    raise RuntimeError('Max retries exceeded')

_raw_gen    = client.models.generate_content
_raw_stream = client.models.generate_content_stream
_raw_embed  = client.models.embed_content
_raw_count  = client.models.count_tokens
client.models.generate_content        = lambda *a,**kw: _call_with_retry(_raw_gen,    *a,**kw)
client.models.generate_content_stream = lambda *a,**kw: _call_with_retry(_raw_stream, *a,**kw)
client.models.embed_content           = lambda *a,**kw: _call_with_retry(_raw_embed,  *a,**kw)
client.models.count_tokens            = lambda *a,**kw: _call_with_retry(_raw_count,  *a,**kw)

print('\u2705 Setup complete. Model:', MODEL, '| Retry-on-429: enabled')


✅ Setup complete. Model: gemma-4-31b-it | Retry-on-429: enabled


### 🔌 API Key Test

In [3]:
try:
    _r = client.models.generate_content(
        model=MODEL,
        contents="Reply with exactly the words: Hello LLM course",
        config=cfg(temperature=0.0)
    )
    print("✅ API key working!")
    print("Model says:", get_text(_r))
except Exception as e:
    print("❌ API key error:", e)

✅ API key working!
Model says: Hello LLM course


---
## 11. Hallucination & Grounding

### What is hallucination?

LLMs **confidently generate false information**. This isn't a bug — it's a consequence of how they work. The model predicts the most *plausible-sounding* tokens, not the most *factually correct* ones.

Common patterns: invented citations, wrong numbers, fabricated events, confident but incorrect technical details.

### Mitigation strategies
| Strategy | How |
|---|---|
| **Lower temperature** | More deterministic = less likely to fabricate |
| **Grounding** | Provide the facts in the prompt; instruct model to use only those |
| **Verification step** | Second prompt: "Is this verifiable? Rate your confidence" |
| **RAG** | Retrieve real documents, include them in the prompt |

In [4]:
# Ask about something that doesn't exist — the model may hallucinate details
r = client.models.generate_content(
    model=MODEL,
    contents=(
        "Who won the fictional 'Golden Syntax Award' for best Python library in 2023? "
        "Give details about the award ceremony."
    ),
    config=cfg(temperature=0.7)
)
print("Hallucination-prone prompt (award doesn't exist):")
print(get_text(r))

Hallucination-prone prompt (award doesn't exist):
Since the **Golden Syntax Award** is fictional, I have taken the liberty of imagining the prestigious (and entirely imaginary) 2023 ceremony for you.

### The Winner
The 2023 Golden Syntax Award for Best Python Library was awarded to **`NebulaPy`**, a fictional library designed to "optimize code by distributing computational loads across simulated galactic clusters." 

`NebulaPy` won the award specifically for its groundbreaking feature, `astral_projection()`, which allegedly allows a developer to debug their code in a parallel dimension where the bugs have already been fixed.

***

### The Ceremony Details
The event was held on November 14, 2023, at the **Recursive Ballroom** in the virtual city of *Pythonia*.

**The Dress Code:**
The dress code was strictly **"PEP 8 Compliant."** Attendees were required to wear outfits with perfect alignment and no unnecessary whitespace. Those who arrived wearing "cluttered" accessories (like mismatc

In [5]:
# Grounding: provide the facts, instruct the model to use only them
context = """
COMPANY POLICY DOCUMENT (Q1 2024):
- Remote work allowance: up to 3 days per week
- Annual leave: 25 days per year
- Health insurance: covers employee and up to 2 dependants
- Equipment budget: $1,500 per year
"""

grounded_prompt = f"""Use ONLY the information in the document below to answer the question.
If the answer is not in the document, say exactly: "This information is not in the provided document."

DOCUMENT:
{context}

QUESTION: How many days of remote work are allowed per week?"""

r_grounded = client.models.generate_content(
    model=MODEL, contents=grounded_prompt,
    config=cfg(temperature=0.0)
)
print("Grounded answer:")
print(get_text(r_grounded))

Grounded answer:
up to 3 days per week


In [6]:
# Test: question NOT in the document — should refuse to answer
grounded_prompt2 = f"""Use ONLY the information in the document below to answer the question.
If the answer is not in the document, say exactly: "This information is not in the provided document."

DOCUMENT:
{context}

QUESTION: What is the company's parental leave policy?"""

r2 = client.models.generate_content(
    model=MODEL, contents=grounded_prompt2,
    config=cfg(temperature=0.0)
)
print("Out-of-scope question:")
print(get_text(r2))

Out-of-scope question:
This information is not in the provided document.


In [7]:
# Verification step: ask the model to rate its own confidence
claim = "The Python programming language was created in 1991 by Guido van Rossum."

r_check = client.models.generate_content(
    model=MODEL,
    contents=f"""Fact-check this claim. Rate confidence as HIGH, MEDIUM, or LOW. Explain any uncertainties.

Claim: {claim}""",
    config=cfg(temperature=0.0)
)
print(get_text(r_check))

**Verdict:** True
**Confidence:** HIGH

**Explanation:**
Guido van Rossum began working on Python in the late 1980s (specifically December 1989) at the Centrum Wiskunde & Informatica (CWI) in the Netherlands. However, the first public release (version 0.9.0) occurred in **February 1991**. In the context of software history, 1991 is the widely accepted date for the language's creation/release.

**Uncertainties:**
There are no significant uncertainties. The only minor nuance is the distinction between when development started (1989) and when it was first released to the public (1991), but the claim is accurate by standard historical accounts.


---
## 12. Context Window Limits

Every model has a **context window** — the maximum tokens it can process (prompt + response combined).

| Model | Context window |
|---|---|
| Gemma 4 31B | ~128k tokens |
| Gemini 2.5 Flash | 1M tokens |

~128k tokens ≈ ~100,000 words ≈ a full novel.

### Strategies for long documents
| Strategy | When to use |
|---|---|
| **Chunking** | Split document, process each chunk separately |
| **Summarization chain** | Summarize each chunk, then summarize the summaries |
| **RAG** | Embed chunks, retrieve only the relevant ones |

In [8]:
# Measure token counts for different document sizes
sentence = "The quick brown fox jumps over the lazy dog. "

print(f"{'~Words':>10} {'Tokens':>10} {'% of 128k window':>20}")
print("-" * 45)
for n_words in [100, 1_000, 10_000, 50_000]:
    text = sentence * (n_words // 9)
    result = client.models.count_tokens(model=MODEL, contents=text)
    tokens = result.total_tokens
    pct = tokens / 128_000 * 100
    print(f"{n_words:>10,} {tokens:>10,} {pct:>19.1f}%")

    ~Words     Tokens     % of 128k window
---------------------------------------------
       100        112                 0.1%


     1,000      1,112                 0.9%


    10,000     11,112                 8.7%


    50,000     55,552                43.4%


In [9]:
# Chunking strategy: split a long text, summarize each chunk, combine
long_text = """
Chapter 1: Python was created by Guido van Rossum in 1991. It emphasizes readability 
and simplicity. The name comes from Monty Python, not the snake.

Chapter 2: Python has a rich ecosystem of libraries. NumPy handles numerical computation,
Pandas is for data manipulation, and Flask/Django are popular web frameworks.

Chapter 3: Python is widely used in data science, machine learning, web development,
automation, and scientific computing. It consistently ranks as one of the most 
popular programming languages worldwide.
"""

chunks = [c.strip() for c in long_text.strip().split("\n\n") if c.strip()]

# Step 1: summarize each chunk
chunk_summaries = []
for i, chunk in enumerate(chunks, 1):
    r = client.models.generate_content(
        model=MODEL,
        contents=f"Summarize in ONE sentence:\n{chunk}",
        config=cfg(temperature=0.0)
    )
    summary = get_text(r)
    chunk_summaries.append(summary)
    print(f"Chunk {i}: {summary}")

# Step 2: combine summaries
r_final = client.models.generate_content(
    model=MODEL,
    contents="Combine these summaries into one paragraph:\n" + "\n".join(chunk_summaries),
    config=cfg(temperature=0.3)
)
print("\nFinal combined summary:")
print(get_text(r_final))

Chunk 1: Created by Guido van Rossum in 1991, Python is a language designed for simplicity and readability and named after Monty Python.


Chunk 2: Python features a rich ecosystem of libraries for numerical computation, data manipulation, and web development, including NumPy, Pandas, Flask, and Django.


Chunk 3: Python is a globally popular and versatile programming language used across various fields, including data science, machine learning, web development, and automation.



Final combined summary:
Created by Guido van Rossum in 1991 and named after Monty Python, Python is a globally popular and versatile programming language designed for simplicity and readability. It is widely used across various fields—including data science, machine learning, automation, and web development—and is supported by a rich ecosystem of libraries such as NumPy, Pandas, Flask, and Django for numerical computation and data manipulation.


---
## 13. Generating Multiple Responses (Best-of-N)

Sometimes you want **N independent responses** to the same prompt — then pick the best one.
This is called **best-of-N sampling** and is a simple but powerful quality-improvement technique.

Why it helps:
- **Diversity** — higher temperature gives varied outputs; choose the one that fits best
- **Reliability** — if most candidates agree, you have more confidence in the answer
- **Evaluation** — compare different phrasings or approaches side by side

> **Note:** Some API providers support `candidate_count` to get multiple responses in one call.
> With Gemma 4 we simply call the API N times — same effect, slightly more tokens.

In [10]:
# Generate 3 different taglines by calling the API 3 times at high temperature
prompt = "Write ONLY a single marketing tagline (no preamble) for a Python learning app called 'PyLeap'."
N = 3

candidates = []
for i in range(N):
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.2)  # high temperature for variety
    )
    candidates.append(get_text(r))

print(f"Generated {N} candidates:\n")
for i, text in enumerate(candidates, 1):
    print(f"Candidate {i}: {text}")

Generated 3 candidates:

Candidate 1: Leap into Python mastery.
Candidate 2: Leap into Python mastery.
Candidate 3: Leap from beginner to pro.


In [11]:
# Best-of-N: generate N subject lines, then use the model to judge the best
prompt = "Write ONLY a single email subject line (no preamble) announcing a new Python course."

candidates_text = []
for _ in range(3):
    r = client.models.generate_content(
        model=MODEL, contents=prompt,
        config=cfg(temperature=1.0)
    )
    candidates_text.append(get_text(r))

print("Candidates:")
for i, t in enumerate(candidates_text, 1):
    print(f"  {i}. {t}")

# Ask the model to pick the best one
selection_prompt = "Which subject line is most compelling? Reply with just the number (1, 2, or 3) and a one-sentence reason.\n\n"
selection_prompt += "\n".join(f"{i+1}. {t}" for i, t in enumerate(candidates_text))

r_pick = client.models.generate_content(
    model=MODEL, contents=selection_prompt,
    config=cfg(temperature=0.0)
)
print(f"\nModel's pick: {get_text(r_pick)}")

Candidates:
  1. 🚀 Now Live: Master Python with Our New Course!
  2. Master Python: Our New Comprehensive Course is Now Live! 🚀
  3. Master Python: Join Our New Course Today! 🐍



Model's pick: 2. It leads with the primary benefit and uses a strong adjective to convey value before the announcement.
